In [3]:
#Figuring and Building the Taxonomy Data

import pandas as pd
import json

# ==========================================
# 1. CONFIGURATION
# ==========================================
FILE_PATH = "/Users/nejam_abderrahmane/Downloads/Split_Files/Mercari.xlsx"  # <--- REPLACE with your actual file name
SHEET_NAME = 0                # Load the first sheet by default

def build_taxonomy_from_excel(file_path):
    # 1. Load the Excel File
    print(f" Loading {file_path}...")
    try:
        df = pd.read_excel(file_path, sheet_name=SHEET_NAME)
    except FileNotFoundError:
        print(" Error: File not found. Please check the file name.")
        return
    
    # 2. Auto-Detect Category Columns
    # We look for columns that contain the word "category" (case-insensitive)
    # OR you can manually set: hierarchy_cols = ["Main_Cat", "Sub_Cat", "Type"]
    hierarchy_cols = ["category_1", "category_2", "category_3"]
    
    # Sort them to ensure order (category_1 -> category_2 -> category_3)
    hierarchy_cols.sort()
    
    if not hierarchy_cols:
        print(" No columns found with 'category' in the name.")
        print(f"Available columns: {list(df.columns)}")
        return

    print(f"✅ Found hierarchy columns: {hierarchy_cols}")
    
    # 3. Build the Tree
    # We only care about unique paths (combinations of categories)
    # dropna() ensures we don't map empty rows
    unique_paths = df[hierarchy_cols].drop_duplicates().dropna(how='all')
    
    taxonomy_tree = {}

    for _, row in unique_paths.iterrows():
        current_level = taxonomy_tree
        
        for col in hierarchy_cols:
            cat_name = row[col]
            
            # Skip if the cell is empty (NaN) or just whitespace
            if pd.isna(cat_name) or str(cat_name).strip() == "":
                continue
            
            cat_name = str(cat_name).strip()
            
            # Create node if it doesn't exist
            if cat_name not in current_level:
                current_level[cat_name] = {}
            
            # Move deeper
            current_level = current_level[cat_name]
            
    return taxonomy_tree

# ==========================================
# 4. RUN AND VISUALIZE
# ==========================================
taxonomy = build_taxonomy_from_excel(FILE_PATH)

if taxonomy:
    print("\n EXTRACTED TAXONOMY TREE:")
    print("===========================")
    
    # Helper function to print tree visually
    def print_tree(node, level=0):
        for key, value in node.items():
            indent = "    " * level
            connector = "└── " if level > 0 else ""
            print(f"{indent}{connector} {key}")
            print_tree(value, level + 1)

    print_tree(taxonomy)
    
    # Optional: Save as JSON file for your AI project
    with open("my_taxonomy.json", "w", encoding="utf-8") as f:
        json.dump(taxonomy, f, ensure_ascii=False, indent=4)
    print("\n✅ Saved taxonomy to 'my_taxonomy.json'")

 Loading /Users/nejam_abderrahmane/Downloads/Split_Files/Mercari.xlsx...
✅ Found hierarchy columns: ['category_1', 'category_2', 'category_3']

 EXTRACTED TAXONOMY TREE:
 ゲーム・おもちゃ・グッズ
    └──  トレーディングカード
        └──  その他
        └──  ポケモンカードゲーム
        └──  タレントカード
        └──  ヴァイスシュヴァルツ
        └──  遊戯王OCG デュエルモンスターズ
        └──  ドラゴンボールカード
        └──  サプライ・アクセサリ・グッズ
        └──  スポーツカード
        └──  シャドウバース エボルヴ
        └──  カードダスその他
        └──  カードファイト!! ヴァンガード
        └──  遊戯王OCG デュエルモンスターズ(外国語版)
        └──  デジモンカードゲーム
        └──  ワンピース カードゲーム
        └──  Z/X (ゼクス)
        └──  ビルディバイド
        └──  ヴァイスシュヴァルツブラウ
        └──  デュエルマスターズ
        └──  ゲームセンター・ゲームカード
        └──  名探偵コナンカードゲーム
        └──  マジック：ザ・ギャザリング(外国語版)
        └──  遊戯王ラッシュデュエル
        └──  UNION ARENA(ユニオンアリーナ)
        └──  ブシロードその他
        └──  エンタメカード
        └──  バトルスピリッツ
        └──  ウィクロス
        └──  マジック：ザ・ギャザリング
        └──  ホロライブ カードダス
        └──  Reバース for you(リバースフォーユー)
        └──  トレーディングカード
  